<a href="https://colab.research.google.com/github/markinkus/AIFeatures/blob/main/Finetuning_LLM-Gemma3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Addestra un LLM sui tuoi dati - Tutorial Simone Rizzo
Questo notebook è una demo per dimostrare come si può addestrare un LLM open-source privato sui propri dati per poi esportarlo in GGFU per Ollama.

**Nota bene**: se vuoi usare la GPU gratuita vai da Modifica -> Impostazioni blocco note -> T4 GPU

La demo è stata realizzata da **Simone Rizzo**:
- [Youtube](https://www.youtube.com/channel/UCbMlkb79E12CwveGAtdFj-A)
- [Linkedin](https://www.linkedin.com/in/simone-rizzo-9851b7147/)
- [TikTok](https://www.tiktok.com/@simonerizzo98)
- [Instagram](https://www.instagram.com/simorizzo_ai/)

Seguimi e lascia un like sui miei social 😜

# Installazione delle librerie

In [2]:
# %%capture
!pip install unsloth
# Also get the latest nightly Unsloth!
!pip uninstall unsloth -y && pip install --upgrade --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git
# Also get the real guide:
!pip uninstall unsloth -y
!pip install bitsandbytes
!pip install git+https://github.com/unslothai/unsloth.git
!pip install unsloth_zoo
!pip install llama-cpp-python --upgrade

Found existing installation: unsloth 2025.5.10
Uninstalling unsloth-2025.5.10:
  Successfully uninstalled unsloth-2025.5.10
  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-req-build-45e4jb_t
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-req-build-45e4jb_t
  Resolved https://github.com/unslothai/unsloth.git to commit beef0cbcb6ecf1fa126589bd2877be85a91bfb8f
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for unsloth: filename=unsloth-2025.5.10-py3-none-any.whl size=275730 sha256=4ada1ebe0d1565fb00d64f9a79b71079972f6f906940365e341e2a3a7403f695
  Stored in directory: /tmp/pip-ephem-wheel-cache-q405fmug/wheels/d1/17/05/850ab10c33284a4763b0595cd8ea9d01fce6e221cac24b3c01
Successfully built unsloth


# Inizializzazione del modello

In [76]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = False # Use 4bit quantization to reduce memory usage. Can be False.

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "google/gemma-3-1b-it", # or choose "unsloth/Llama-3.2-1B-Instruct"
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = "hf_...", # use one if using gated models like meta-llama/Llama-2-7b-hf
)

==((====))==  Unsloth 2025.5.10: Fast Gemma patching. Transformers: 4.52.4.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


OSError: google/gemma-3-1b-it-qat-q4_0-gguf does not appear to have a file named pytorch_model.bin, model.safetensors, tf_model.h5, model.ckpt or flax_model.msgpack.

In [4]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth: Making `model.base_model.model.model` require gradients


In [6]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "gemma-3",
)
FastLanguageModel.for_inference(model) # Enable native 2x faster inference

messages = [
    {"role": "user", "content": "Spiegami cosa è BullVerge, cosa è prometeo e da chi è stato costruito"},
]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True, # Must add for generation
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(input_ids = inputs, streamer = text_streamer, max_new_tokens = 128,
                   use_cache = True, temperature = 0.7, min_p = 0.1)

Certamente! BullVerge è un progetto di **blockchain di alto livello** che mira a creare un'infrastruttura per la **gestione e l'interoperabilità di dati** in un'era di crescente complessità e frammentazione.  È un progetto molto ambizioso e in continua evoluzione.

**Cosa è BullVerge?**

Immagina un sistema che non si limita a memorizzare dati, ma che li **organizza, li riassume, li analizza e li rende accessibili in modo intelligente**. BullVerge si propone di fare proprio questo, offrendo una


# Formattazione del testo per il formato di LLama3.2

In [7]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "gemma-3",
)

In [8]:
import pandas as pd
df = pd.read_csv("/content/sflux-training-dataset.csv")

In [9]:
df.head()

,User,Prompt
0,"""Cos'è esattamente BullVerge?""","""BullVerge è un progetto italiano che mira a d..."
1,"""Ho sentito parlare di Prometeo. Potete dirmi ...","""Prometeo è il dispositivo hardware di punta d..."
2,"""Cosa significa Edge AI"" nel contesto di BullV...","""Per BullVerge Edge AI"" significa che le elabo..."
3,"""Qual è il sistema operativo di Prometeo?""","""Prometeo utilizza BullVerge_OS un sistema ope..."
4,"""Perché BullVerge punta sull'elaborazione loca...","""BullVerge enfatizza l'elaborazione locale del..."


In [10]:
df = df.dropna()

In [11]:
from datasets import Dataset
df["conversations"] = df.apply(
    lambda x: [
        {"content": x["User"], "role": "user"},
        {"content": x["Prompt"], "role": "assistant"}
    ], axis=1
)

# Ora convertiamo il DataFrame in un Dataset di HuggingFace, rimuovendo le vecchie colonne
dataset = Dataset.from_pandas(df.drop(columns=["User", "Prompt"]))

In [12]:
dataset

Dataset({
    features: ['conversations', '__index_level_0__'],
    num_rows: 423
})

In [13]:
dataset['conversations'][0]

[{'content': '"Cos\'è esattamente BullVerge?"', 'role': 'user'},
 {'content': '"BullVerge è un progetto italiano che mira a democratizzare l\'intelligenza artificiale rendendola accessibile a tutti. Sviluppano soluzioni AI che funzionano localmente sui dispositivi senza dipendere dal cloud con un forte focus sulla privacy la sostenibilità e il controllo da parte dell\'utente. Il loro slogan è Beyond AImagination"."""',
  'role': 'assistant'}]

In [14]:
def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
    return { "text" : texts, }
pass

from unsloth.chat_templates import standardize_sharegpt
dataset = standardize_sharegpt(dataset)
dataset = dataset.map(formatting_prompts_func, batched = True,)

Unsloth: Standardizing formats (num_proc=2):   0%|          | 0/423 [00:00<?, ? examples/s]

Map:   0%|          | 0/423 [00:00<?, ? examples/s]

In [15]:
dataset[5]["conversations"]

[{'content': '"Quali sono i valori fondamentali di BullVerge?"',
  'role': 'user'},
 {'content': '"I valori fondamentali di BullVerge includono la democratizzazione dell\'IA la sostenibilità la privacy e il controllo dei dati da parte dell\'utente. Credono in un\'IA che sia uno strumento al servizio delle persone accessibile e rispettosa."',
  'role': 'assistant'}]

In [16]:
dataset[5]["text"]

'<bos><start_of_turn>user\n"Quali sono i valori fondamentali di BullVerge?"<end_of_turn>\n<start_of_turn>model\n"I valori fondamentali di BullVerge includono la democratizzazione dell\'IA la sostenibilità la privacy e il controllo dei dati da parte dell\'utente. Credono in un\'IA che sia uno strumento al servizio delle persone accessibile e rispettosa."<end_of_turn>\n'

# Addestramento del modello

In [22]:
def convert_conversations_to_text(example):
    """
    Converte il formato conversazionale in testo lineare
    che il trainer può processare correttamente
    """
    # Assumendo che tu abbia una struttura come 'conversations'
    if 'conversations' in example:
        conversations = example['conversations']

        # Costruiamo il testo concatenando tutti i messaggi
        text_parts = []

        for message in conversations:
            # Gestisce diversi formati possibili
            if isinstance(message, dict):
                if 'from' in message and 'value' in message:
                    # Formato standard ChatML
                    role = message['from']
                    content = message['value']

                    # Usa token speciali per distinguere i ruoli
                    if role in ['human', 'user']:
                        text_parts.append(f"<|user|>\n{content}")
                    elif role in ['gpt', 'assistant']:
                        text_parts.append(f"<|assistant|>\n{content}")
                    else:
                        text_parts.append(f"<|{role}|>\n{content}")

                elif 'role' in message and 'content' in message:
                    # Formato OpenAI
                    role = message['role']
                    content = message['content']
                    text_parts.append(f"<|{role}|>\n{content}")

        # Unisce tutto in un singolo testo
        full_text = "\n".join(text_parts) + "<|end|>"

        return {"text": full_text}

    # Se il dataset ha già il campo 'text', lo restituisce così com'è
    elif 'text' in example:
        return example

    else:
        raise ValueError("Dataset non ha né 'conversations' né 'text' field")

# Applica la trasformazione al tuo dataset
print("Trasformando il dataset dal formato conversazionale...")
processed_dataset = dataset.map(
    convert_conversations_to_text,
    remove_columns=[col for col in dataset.column_names if col != 'text'],
    desc="Processando conversazioni"
)

# Verifica che la trasformazione sia andata a buon fine
print(f"Esempio trasformato: {processed_dataset[0]['text'][:200]}...")

Trasformando il dataset dal formato conversazionale...


Processando conversazioni:   0%|          | 0/423 [00:00<?, ? examples/s]

Esempio trasformato: <|user|>
"Cos'è esattamente BullVerge?"
<|assistant|>
"BullVerge è un progetto italiano che mira a democratizzare l'intelligenza artificiale rendendola accessibile a tutti. Sviluppano soluzioni AI che...


In [25]:
from trl import SFTTrainer
from transformers import TrainingArguments, DataCollatorForSeq2Seq
from unsloth import is_bfloat16_supported
# Rimuovi l'import di train_on_responses_only se non la usi altrove
# from unsloth.chat_templates import train_on_responses_only

# Add this line to print a sample of the formatted text
print("Sample formatted text:", dataset[0]["text"])
# You can also print more samples if needed
# print("Sample formatted text:", dataset[1]["text"])

from transformers import DataCollatorForLanguageModeling

# Usa il dataset processato invece di quello originale
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=processed_dataset,  # Dataset trasformato
    dataset_text_field="text",  # Ora sappiamo che esiste
    max_seq_length=max_seq_length,

    # Data collator più appropriato per il fine-tuning conversazionale
    data_collator=DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=False,  # Non masked language modeling
        pad_to_multiple_of=8  # Ottimizzazione per GPU
    ),

    dataset_num_proc=2,
    packing=True,  # Ora può funzionare correttamente

    args=TrainingArguments(
        per_device_train_batch_size=4,
        gradient_accumulation_steps=8,

        # Warmup più appropriato per i tuoi 80 steps
        warmup_steps=20,  # 25% del training totale
        max_steps=80,
        learning_rate=5e-5,

        # Ottimizzazioni numeriche
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),

        # Monitoraggio migliorato
        logging_steps=10,
        save_steps=40,

        # Ottimizzatore e regolarizzazione
        optim="adamw_torch_fused",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        max_grad_norm=1.0,

        # Gestione memoria
        dataloader_pin_memory=True,
        remove_unused_columns=True,  # Ora possiamo rimuovere colonne extra

        seed=3407,
        output_dir="outputs",
        report_to="none",
    )
)

# Rimuovi completamente la chiamata a train_on_responses_only
# trainer = train_on_responses_only(
#     trainer,
#     instruction_part = instruction_part_to_match,
#     response_part = response_part_to_match,
# )

Sample formatted text: <bos><start_of_turn>user
"Cos'è esattamente BullVerge?"<end_of_turn>
<start_of_turn>model
"BullVerge è un progetto italiano che mira a democratizzare l'intelligenza artificiale rendendola accessibile a tutti. Sviluppano soluzioni AI che funzionano localmente sui dispositivi senza dipendere dal cloud con un forte focus sulla privacy la sostenibilità e il controllo da parte dell'utente. Il loro slogan è Beyond AImagination"."""<end_of_turn>

Unsloth: Switching to float32 training since model cannot work with float16


Unsloth: Tokenizing ["text"]:   0%|          | 0/423 [00:00<?, ? examples/s]

In [26]:
#@title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.741 GB.
2.471 GB of memory reserved.


In [30]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 423 | Num Epochs = 6 | Total steps = 80
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 8 x 1) = 32
 "-____-"     Trainable parameters = 13,045,760/1,012,931,712 (1.29% trained)


Step,Training Loss
10,1.927700
20,1.825900
30,1.767700
40,1.702700
50,1.665300
60,1.638200
70,1.592900
80,1.616900


# Inferenza con Streaming

In [33]:
FastLanguageModel.for_inference(model) # Enable native 2x faster inference

messages = [
    {"role": "user", "content": "Spiegami cosa è BullVerge, cosa è prometeo e da chi è stato costruito"},
]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True, # Must add for generation
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(input_ids = inputs, streamer = text_streamer, max_new_tokens = 128,
                   use_cache = True, temperature = 0.5, min_p = 0.1)

Vergognare BullVerge è un concetto in evoluzione e un progetto di ultima generazione che sta suscitando grande interesse nel mondo dell'IA decentralizzata e della blockchain. Cerchiamo di capire cosa è BullVerge e cosa lo rende speciale.

**Cos'è BullVerge?**

In termini semplici BullVerge è un progetto open-source che mira a creare un sistema di interazione AI-Verge (Verge AI) che opera in modo decentralizzato e resistente alla censura. Non è una vera e propria azienda ma un ecosistema in cui Prometeo (il nome del BV Lab)


# Esporto in GGUF per Ollama

In [85]:
from unsloth.save import save_to_gguf_generic
# model = model.merge_and_unload() # Gemma 3 non ha merge_and_unload perchè non è LORA ready;
# 1. Salva modello e tokenizer in formato HuggingFace
model.save_pretrained("merged_model")
tokenizer.save_pretrained("merged_model")

# #2. scarica il tokenizer.model da https://huggingface.co/google/gemma-2b/tree/main e mettilo in merged_model

# 3. Salva in GGUF
save_to_gguf_generic(
    save_directory="merged_model",
    quantization_type="q8_0",
    model=model,
)


Unsloth GGUF:hf-to-gguf:Loading model: merged_model
Unsloth GGUF:hf-to-gguf:Model architecture: Gemma3ForCausalLM
Unsloth GGUF:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
Unsloth GGUF:hf-to-gguf:Exporting model...
Unsloth GGUF:hf-to-gguf:gguf: loading model part 'model.safetensors'
Unsloth GGUF:hf-to-gguf:token_embd.weight,                 torch.float16 --> Q8_0, shape = {1152, 262144}
Unsloth GGUF:hf-to-gguf:output_norm.weight,                torch.float32 --> F32, shape = {1152}
Unsloth GGUF:hf-to-gguf:Set meta model
Unsloth GGUF:hf-to-gguf:Set model parameters
Unsloth GGUF:hf-to-gguf:Set model quantization version
Unsloth GGUF:hf-to-gguf:Set model tokenizer
Unsloth GGUF:gguf.vocab:Setting special token type bos to 2
Unsloth GGUF:gguf.vocab:Setting special token type eos to 106
Unsloth GGUF:gguf.vocab:Setting special token type unk to 3
Unsloth GGUF:gguf.vocab:Setting special token type pad to 0
Unsloth GGUF:gguf.vocab:Setting add_bos_token to True
Unsloth GGUF:gg

Unsloth: GGUF conversion:   0%|          | 0/100 [00:00<?, ?it/s]

Unsloth GGUF:hf-to-gguf:Model successfully exported to ./
Unsloth: Converted to merged_model.Q8_0.gguf with size = 1.1G
Unsloth: Successfully saved GGUF to:
merged_model.Q8_0.gguf


['merged_model.Q8_0.gguf']

In [86]:
from google.colab import files
files.download('/content/merged_model.Q8_0.gguf')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [71]:
help(model.save_pretrained_gguf)

Help on method save_to_gguf_generic in module unsloth.save:

save_to_gguf_generic(save_directory, quantization_type='Q8_0', repo_id=None, token=None) method of transformers.models.gemma3.modeling_gemma3.Gemma3ForCausalLM instance

